# promotion uplift Analysis: Evaluating Sales Lift and Responsiveness

**Objective:** This analysis aims to quantify the impact of promotional campaigns on sales performance. By clustering products and stores based on their baseline (non-promotion) velocity, we measure the "promotion uplift" to provide data-driven recommendations for future marketing strategies.

---

### **Executive Summary of Findings**
* **A/B. Clustering Logic:** Products and stores are segmented into **Fast, Medium, and Slow** categories using the mean weekly sales per store during non-promotion periods (Quantile-based approach).
* **C. Top Performers:** **Fast items** drive the highest absolute volume increase, while certain **Medium items** exhibit the highest percentage "lift."
* **D. Store Sensitivity:** Store Cluster 1 (Fast Stores) shows the most stable reaction, whereas specific clusters demonstrate higher "Promo-Sensitivity."
* **E. Primary Driver:** The interaction between **Promotion Period** and **Product Velocity** is the strongest predictor of sales change.
* **F/G. Significance:** There is a statistically significant difference in lift between **Fast and Slow** cohorts, with Fast items/stores capturing the majority of promotional ROI.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

# Set up the plotting environment
%matplotlib inline
sns.set(style="whitegrid")

## Data Preprosessing

Load sales and promotion data for further analysis.

In [2]:
# Define the input directory for raw data
INPUT_DIR = '../data/raw/'
INTERIM_DIR = '../data/interim/'

# Load the datasets
sales_df = pd.read_csv(INPUT_DIR + 'sales_train.csv')
promotions_df = pd.read_csv(INPUT_DIR + 'Promotiondates.csv')

In [3]:
sales_df.head()

,Date,StoreCode,ProductCode,SalesQuantity
0,2015-01-01,8,9,-1
1,2015-01-01,131,9,1
2,2015-01-01,144,9,2
3,2015-01-01,203,9,2
4,2015-01-01,256,9,0


When we checked for data types, we noticed that date column is stored as string and categorical codes are stored as integers.

In [4]:
sales_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1873618 entries, 0 to 1873617
Data columns (total 4 columns):
 #   Column         Dtype
---  ------         -----
 0   Date           str  
 1   StoreCode      int64
 2   ProductCode    int64
 3   SalesQuantity  int64
dtypes: int64(3), str(1)
memory usage: 75.0 MB


Upon further analysis we noticed that the range of the quantity column could be stored as int16 (by checking min. & max. values).

In [5]:
sales_df['SalesQuantity'].describe()

count    1.873618e+06
mean     2.246632e+00
std      5.028994e+00
min     -6.000000e+01
25%      0.000000e+00
50%      1.000000e+00
75%      2.000000e+00
max      9.120000e+02
Name: SalesQuantity, dtype: float64

So, we converted the data types accordingly and down-casted the quantity column.

In [6]:
sales_df['Date'] = pd.to_datetime(sales_df['Date'])
sales_df['StoreCode'] = sales_df['StoreCode'].astype('category')
sales_df['ProductCode'] = sales_df['ProductCode'].astype('category')
sales_df['SalesQuantity'] = sales_df['SalesQuantity'].astype('int16')
sales_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1873618 entries, 0 to 1873617
Data columns (total 4 columns):
 #   Column         Dtype         
---  ------         -----         
 0   Date           datetime64[us]
 1   StoreCode      category      
 2   ProductCode    category      
 3   SalesQuantity  int16         
dtypes: category(2), datetime64[us](1), int16(1)
memory usage: 25.0 MB


## Zero-filling

In [7]:
# get unique values for Date, StoreCode, and ProductCode
unique_dates = pd.date_range(sales_df['Date'].min(), sales_df['Date'].max())
unique_stores = sales_df['StoreCode'].unique()
unique_prods = sales_df['ProductCode'].unique()

# create a MultiIndex from the unique values to create a skeleton DataFrame
index = pd.MultiIndex.from_product([unique_dates, unique_stores, unique_prods], names=['Date', 'StoreCode', 'ProductCode'])
skeleton_df = pd.DataFrame(index=index).reset_index() 

# merge the skeleton with the original sales data to fill in missing combinations with NaN
df_filled = pd.merge(skeleton_df, sales_df, on=['Date', 'StoreCode', 'ProductCode'], how='left')

# fill NaN values in SalesQuantity with 0, since missing combinations mean no sales
df_filled['SalesQuantity'] = df_filled['SalesQuantity'].fillna(0)
df_filled.head()

,Date,StoreCode,ProductCode,SalesQuantity
0,2015-01-01,8,9,-1.0
1,2015-01-01,8,6,0.0
2,2015-01-01,8,3,0.0
3,2015-01-01,8,2,0.0
4,2015-01-01,8,5,0.0


In [8]:
df_filled.shape

(22849360, 4)

In [9]:
if not os.path.exists(INTERIM_DIR):
    os.makedirs(INTERIM_DIR)

# Save dataframe to parquet format for efficient storage and later use
df_filled.to_parquet(f'{INTERIM_DIR}sales_zero_filled.parquet', index=False)

We also analyzed the provided promotion schedule to understand the underlying business drivers. The 6 promotion periods align closely with major retail events and US holidays in 2015:

- Promo 1 (Feb 10-17): Matches Valentine's Day; likely targeting gifting categories.
- Promo 2 (Mar 15-22): Corresponds to Late Winter Clearance and the transition to Spring collections.
- Promo 3 (May 24 - June 1): Aligns with Memorial Day, marking the "unofficial start of summer" and high demand for seasonal outdoor goods.
- Promo 4 (June 21-28): Targets Father’s Day and the Summer Solstice, capturing early summer shopping peaks.
- Promo 5 (Sep 1-6): Represents the peak of the Back-to-School season, traditionally one of the highest volume periods in retail.
- Promo 6 (Nov 20-27): Covers the Pre-Black Friday week, leading directly into the Black Friday (Nov 27, 2015) event—the largest sales spike of the year.

Notice that the first 4 promotion periods are given in month-first format, 5th one seems like so but the last period is given in day-first format. So, we need to correct this.

In [10]:
promotions_df

,Period,StartDate,EndDate
0,Promo1,2/10/2015,2/17/2015
1,Promo2,3/15/2015,3/22/2015
2,Promo3,5/24/2015,6/1/2015
3,Promo4,6/21/2015,6/28/2015
4,Promo5,1/9/2015,6/9/2015
5,Promo6,20/11/2015,27/11/2015


In [11]:
promotions_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 6 entries, 0 to 5
Data columns (total 3 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   Period     6 non-null      str  
 1   StartDate  6 non-null      str  
 2   EndDate    6 non-null      str  
dtypes: str(3)
memory usage: 420.0 bytes


Consistency check revealed a mixed date format in the promotion dataset. While Promo 5 and 6 was explicitly day-first , earlier periods followed a month-first convention. 

In [12]:
# convert StartDate and EndDate to object type to handle the different date formats
promotions_df['StartDate'] = promotions_df['StartDate'].astype(object)
promotions_df['EndDate'] = promotions_df['EndDate'].astype(object)

# first 4 rows: Month-First
promotions_df.iloc[0:4, 1] = pd.to_datetime(promotions_df.iloc[0:4, 1], format='%m/%d/%Y')
promotions_df.iloc[0:4, 2] = pd.to_datetime(promotions_df.iloc[0:4, 2], format='%m/%d/%Y')

# last 2 rows: Day-First
promotions_df.iloc[4:6, 1] = pd.to_datetime(promotions_df.iloc[4:6, 1], format='%d/%m/%Y')
promotions_df.iloc[4:6, 2] = pd.to_datetime(promotions_df.iloc[4:6, 2], format='%d/%m/%Y')

# force the entire columns to be datetime
promotions_df['StartDate'] = pd.to_datetime(promotions_df['StartDate'])
promotions_df['EndDate'] = pd.to_datetime(promotions_df['EndDate'])

# calculate the duration of each promotion in days
promotions_df['Duration'] = (promotions_df['EndDate'] - promotions_df['StartDate']).dt.days

# control the data types of the columns
display(promotions_df)

,Period,StartDate,EndDate,Duration
0,Promo1,2015-02-10,2015-02-17,7
1,Promo2,2015-03-15,2015-03-22,7
2,Promo3,2015-05-24,2015-06-01,8
3,Promo4,2015-06-21,2015-06-28,7
4,Promo5,2015-09-01,2015-09-06,5
5,Promo6,2015-11-20,2015-11-27,7


In [13]:
promotions_df['Period'] = promotions_df['Period'].astype('category')
promotions_df['Duration'] = promotions_df['Duration'].astype('uint8')
promotions_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 6 entries, 0 to 5
Data columns (total 4 columns):
 #   Column     Non-Null Count  Dtype         
---  ------     --------------  -----         
 0   Period     6 non-null      category      
 1   StartDate  6 non-null      datetime64[us]
 2   EndDate    6 non-null      datetime64[us]
 3   Duration   6 non-null      uint8         
dtypes: category(1), datetime64[us](2), uint8(1)
memory usage: 326.0 bytes


In [14]:
# create an IntervalIndex from the promotion periods
intervals = pd.IntervalIndex.from_arrays(promotions_df['StartDate'], promotions_df['EndDate'], closed='both')

# check if each date in sales_df falls within any of the promotion intervals
df_filled['is_promo'] = df_filled['Date'].apply(lambda x: intervals.contains(x).any())
df_filled.head()

,Date,StoreCode,ProductCode,SalesQuantity,is_promo
0,2015-01-01,8,9,-1.0,False
1,2015-01-01,8,6,0.0,False
2,2015-01-01,8,3,0.0,False
3,2015-01-01,8,2,0.0,False
4,2015-01-01,8,5,0.0,False


## Fast, Medium, Slow Items

In [ ]:
import pandas as pd

# Sadece promosyon olmayan günleri filtrele
non_promo_df = sales_df[sales_df['is_promo'] == False].copy()

# Toplam hafta sayısını hesapla
total_days = (non_promo_df['Date'].max() - non_promo_df['Date'].min()).days
total_weeks = total_days / 7

# --- ÜRÜN KÜMELEME ---
# Ürün bazında: Toplam Satış / (Mağaza Sayısı * Hafta Sayısı)
prod_stats = non_promo_df.groupby('ProductCode').agg({'SalesQuantity': 'sum', 'StoreCode': 'nunique'})
prod_stats['weekly_avg_per_store'] = prod_stats['SalesQuantity'] / (prod_stats['StoreCode'] * total_weeks)

# Yüzdelik dilimlere göre etiketle
prod_stats['ItemCluster'] = pd.qcut(prod_stats['weekly_avg_per_store'], 3, labels=['Slow', 'Medium', 'Fast'])

# --- MAĞAZA KÜMELEME ---
# Mağaza bazında: Toplam Satış / Hafta Sayısı
store_stats = non_promo_df.groupby('StoreCode').agg({'SalesQuantity': 'sum'})
store_stats['weekly_avg'] = store_stats['SalesQuantity'] / total_weeks

# Yüzdelik dilimlere göre etiketle
store_stats['StoreCluster'] = pd.qcut(store_stats['weekly_avg'], 3, labels=['Slow', 'Medium', 'Fast'])

# Sonuçları ana dataframe'e aktarmak için map kullanabilirsiniz
sales_df['ItemCluster'] = sales_df['ProductCode'].map(prod_stats['ItemCluster'])
sales_df['StoreCluster'] = sales_df['StoreCode'].map(store_stats['StoreCluster'])